# Getting started, one step at a time

Run each cell in order and inspect its output before continuing. The lab handles metadata, raw light-curve loading and batching. You write the labels, preprocessing, model and training loop.

The labels and tiny model here demonstrate the mechanics, not a scientifically validated planet detector.

## 1. Load your settings

Run this notebook from the repository root or the notebooks directory.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
load_dotenv(root / ".env")

## 2. Load the light-curve manifest

This reads metadata from PostgreSQL, not the light curves themselves. To refresh the catalogues and manifest separately, run `uv run --env-file .env astro-data-lab sync` in a terminal.

In [ ]:
from astro_data_lab import load_curves

curves = load_curves()

In [ ]:
print("Columns:", curves.columns.tolist())
print("First curve:", curves.iloc[0].to_dict())
print(f"{len(curves):,} curves from {curves.tic_id.nunique():,} stars")

## 3. Load the TOI catalogue

Each TOI has a TIC star identifier and a catalogue disposition. These are catalogue labels, not proof that a transit is visible in every sector.

In [ ]:
from astro_data_lab import load_tois

tois = load_tois()

In [ ]:
print("Columns:", tois.columns.tolist())
print("Disposition counts:", tois.disposition.value_counts().to_dict())

## 4. Choose the demo positive stars

For this walkthrough only: PC, CP and KP TOIs are the positive class. This does not check transit timing or detectability.

In [ ]:
positive_dispositions = ["PC", "CP", "KP"]
planet_stars = set(tois.loc[tois.disposition.isin(positive_dispositions), "tic_id"])
print(f"{len(planet_stars):,} demo positive stars")

## 5. Find all catalogued stars

We exclude other catalogued stars from the demo negative class.

In [ ]:
from astro_data_lab import load_confirmed_planets

confirmed = load_confirmed_planets()
catalogued = set(tois.tic_id) | set(confirmed.tic_id)
print(f"{len(catalogued):,} catalogued stars")

## 6. Attach the demo labels

1 = planet TOI, 0 = not in either catalogue, -1 = exclude. Uncatalogued does **not** mean signal-free: these negatives may contain planets, binaries or other variability. Replace these rules before interpreting any score scientifically.

In [ ]:
import numpy as np

labelled = curves.copy()
labelled["label"] = np.where(
    labelled.tic_id.isin(planet_stars), 1,
    np.where(labelled.tic_id.isin(catalogued), -1, 0),
)
labelled = labelled.loc[labelled.label >= 0].copy()
print("Label counts:", labelled.label.value_counts().to_dict())

## 7. Select training curves

The existing split is deterministic per star. Different sectors of the same star cannot cross splits. Take 200 curves per class for this small demo.

In [ ]:
train_table = (
    labelled.loc[labelled.split == "train"]
    .groupby("label").sample(200, random_state=0)
    .reset_index(drop=True)
)
print("Training labels:", train_table.label.value_counts().to_dict())

## 8. Select validation curves

Use validation for development. Leave the test split untouched.

In [ ]:
val_table = (
    labelled.loc[labelled.split == "validation"]
    .groupby("label").sample(100, random_state=0)
    .reset_index(drop=True)
)
assert set(train_table.tic_id).isdisjoint(val_table.tic_id)
print("Validation labels:", val_table.label.value_counts().to_dict())

## 9. Load one raw light curve

Input: a manifest URI. Output: a `LightCurve` with raw arrays. Downloads are cached; NaNs and quality-flagged readings remain.

In [ ]:
from astro_data_lab import read_lightcurve

uri = train_table.loc[train_table.label == 1, "data_uri"].iloc[0]
lc = read_lightcurve(uri)
print(f"TIC {lc.tic_id}, sector {lc.sector}")
print("Time shape:", lc.time.shape, "Flux shape:", lc.flux.shape)
print("Non-finite flux:", (~np.isfinite(lc.flux)).sum())
print("Flagged readings:", (lc.quality != 0).sum())

## 10. Plot the raw flux

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 3))
plt.plot(lc.time, lc.flux, ",k")
plt.xlabel("Time (BTJD days)")
plt.ylabel("PDCSAP flux (electrons / s)")
plt.show()

## 11. Define your preprocessing

Input: one `LightCurve`. Output: a one-dimensional array. This example removes flagged/non-finite readings and normalises by the median. It does not detrend or resample; removing readings also hides time gaps from this simple model.

In [ ]:
def transform(lc):
    keep = (lc.quality == 0) & np.isfinite(lc.flux) & np.isfinite(lc.time)
    flux = lc.flux[keep]
    if flux.size == 0:
        raise ValueError(f"No usable readings: {lc.data_uri}")
    median = np.median(flux)
    if not np.isfinite(median) or median == 0:
        raise ValueError(f"Cannot normalise flux: {lc.data_uri}")
    return flux / median - 1

## 12. Try preprocessing on that curve

In [ ]:
processed = transform(lc)
print("Processed shape:", processed.shape)
plt.figure(figsize=(12, 3))
plt.plot(processed, ",k")
plt.xlabel("Retained reading index (not elapsed time)")
plt.ylabel("Relative flux")
plt.show()

## 13. Create the datasets

Construction stores URIs and labels. Loading and preprocessing happen when an item is requested.

In [ ]:
from astro_data_lab import LightCurveDataset

train_dataset = LightCurveDataset(train_table, transform)
val_dataset = LightCurveDataset(val_table, transform)
print("Dataset sizes:", len(train_dataset), len(val_dataset))

## 14. Inspect one dataset item

Output: a float tensor and an integer label. Unavailable curves return `None`; collation drops those items and raises if an entire batch is unavailable.

In [ ]:
item = train_dataset[0]
if item is None:
    raise RuntimeError("The example curve was unavailable; inspect its URI before continuing")
example, label = item
print("Tensor shape:", tuple(example.shape), "dtype:", example.dtype, "label:", label)

## 15. Create the training DataLoader

Eight workers load curves in parallel. `pad_collate` pads variable lengths and supplies a mask.

In [ ]:
from torch.utils.data import DataLoader
from astro_data_lab import pad_collate

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True,
    num_workers=8, collate_fn=pad_collate,
)

## 16. Create the validation DataLoader

In [ ]:
val_loader = DataLoader(
    val_dataset, batch_size=32,
    num_workers=8, collate_fn=pad_collate,
)

## 17. Inspect one batch

`x`: (batch, channels, readings). `mask`: (batch, readings), True for real data. `y`: (batch,).

In [ ]:
x, mask, y = next(iter(train_loader))
print("x:", tuple(x.shape))
print("mask:", tuple(mask.shape))
print("y:", y.tolist())
print("Real lengths:", mask.sum(dim=1).tolist())

## 18. Inspect the cache

The loader reuses downloaded arrays and evicts old files periodically according to `LIGHTCURVE_CACHE_GIB`. The cap is approximate, not a strict per-batch limit.

**You can stop here.** You now have the data plumbing. The remaining cells are an optional, ordinary PyTorch example, not a hidden training framework.

In [ ]:
from astro_data_lab import cache_info

print(cache_info())

## 19. Define a tiny model

Input: a batch and its mask. Output: two class scores per curve. This is a mechanics example, not an architecture recommendation.

In [ ]:
import torch
from torch import nn

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(1, 16, kernel_size=61, padding=30)
        self.head = nn.Linear(16, 2)

    def forward(self, x, mask):
        features = torch.relu(self.conv(x * 100))
        features = features.masked_fill(~mask.unsqueeze(1), 0)
        return self.head(features.amax(dim=-1))

## 20. Create the model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TinyModel().to(device)
print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## 21. Check a forward pass

In [ ]:
with torch.no_grad():
    scores = model(x.to(device), mask.to(device))
print("Scores shape:", tuple(scores.shape))

## 22. Define validation prediction

Output: two lists, the true labels and predicted labels.

In [ ]:
def predict(model, loader):
    model.eval()
    truth, predicted = [], []
    with torch.no_grad():
        for x, mask, y in loader:
            scores = model(x.to(device), mask.to(device))
            predicted.extend(scores.argmax(dim=1).tolist())
            truth.extend(y.tolist())
    return truth, predicted

## 23. Create the optimiser

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

## 24. Configure MLflow

The next cell contacts your MLflow server. Logging uses MLflow directly, not a lab wrapper.

In [ ]:
import os
import mlflow

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000"))
mlflow.set_experiment("getting-started")

## 25. Train five epochs

This cell actually trains and logs a run. Each batch loads through your Dataset and calls your transform. The loop stays here so you can read and change it.

In [ ]:
from astro_data_lab import report

with mlflow.start_run(run_name="tiny-model"):
    mlflow.log_params({"filters": 16, "lr": 1e-3, "train_curves": len(train_dataset)})
    for epoch in range(5):
        model.train()
        total_loss = 0.0
        seen = 0
        for x, mask, y in train_loader:
            scores = model(x.to(device), mask.to(device))
            loss = nn.functional.cross_entropy(scores, y.to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)
            seen += len(y)

        truth, predicted = predict(model, val_loader)
        accuracy = report(truth, predicted).balanced_accuracy
        train_loss = total_loss / seen
        mlflow.log_metrics({
            "train_loss": train_loss,
            "val_balanced_accuracy": accuracy,
        }, step=epoch)
        print(f"Epoch {epoch + 1}: loss={train_loss:.3f}, validation balanced accuracy={accuracy:.3f}")

## 26. Inspect validation performance

Always predicting one class gives balanced accuracy 0.5 when both classes are present. This interval resamples curves, not stars, and does not account for label errors or repeated model selection. A score above chance does not establish a reliable transit detector.

In [ ]:
truth, predicted = predict(model, val_loader)
print(report(truth, predicted, ["uncatalogued", "planet TOI"]))

Next: agree on defensible class rules, inspect more raw examples, then change preprocessing or the model one thing at a time. Larger selections use the same Dataset and DataLoader; no separate model or trainer package is required.